<a href="https://colab.research.google.com/github/rusisg/bombo/blob/main/Lab01_DSSBD_PostgreSQL_GColab_EN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Laboratory Work No. 1
## Installing PostgreSQL and Creating the First Server Database

**Course:** Design and Security of Server Databases  
**Topic:** Installing PostgreSQL. Creating the First Server Database  
**Environment:** Google Colab

**Full Name:** ____________________________________  
**Group:** ________________________________________  
**Date:** _________________________________________

### Objective
To master the basic operations of deploying the PostgreSQL server DBMS, starting the server, connecting to it, creating a database, tables, and users, and executing SQL queries from `psql` and Python.

### After completing this laboratory work, the student should be able to
- distinguish between a DBMS server, a database, and a client;
- install and start PostgreSQL;
- create a database and tables;
- add, modify, and retrieve data;
- create a separate database user and assign privileges;
- connect to PostgreSQL from Python;
- explain the `host`, `port`, `database`, `user`, and `password` parameters.

> **Important:** Google Colab is a temporary learning environment. After the Runtime is reset, PostgreSQL and the created data may be deleted.

## 1. Checking the Runtime Environment

In [ ]:
!cat /etc/os-release
!uname -a

### Question 1:
1.1 Which operating system is being used?  
**Answer:** _______________________________________________

1.2 What is the difference between an **Operating System**, **Database Server**, and **Database Client**?  
**Answer:**  
__________________________________________________________________  
__________________________________________________________________

## 2. Installing PostgreSQL

In [ ]:
!apt-get update -qq
!DEBIAN_FRONTEND=noninteractive apt-get install -y postgresql postgresql-contrib
!psql --version

## 3. Starting PostgreSQL Server

In [ ]:
!service postgresql start

In [ ]:
!pg_lsclusters
!pg_isready

### Please Note
- the standard PostgreSQL port is **5432**;
- `postgres` is the PostgreSQL administrative role;
- `main` is a typical PostgreSQL cluster name in Debian/Ubuntu.

## 4. Checking the Server and Version

In [ ]:
!timeout 10s sudo -u postgres psql -X -d postgres -c "SELECT version();"

In [ ]:
!timeout 10s sudo -u postgres psql -X -d postgres -c \
"SELECT current_user, current_database(), inet_server_port();"

In [ ]:
!timeout 10s sudo -u postgres psql -X -d postgres -c "SHOW port;"

## 5. Viewing Existing Databases

In [ ]:
!timeout 10s sudo -u postgres psql -c "\l"

### Question 2:
Which standard databases do you see? What are `postgres`, `template0`, and `template1` used for?

**Answer:**  
__________________________________________________________________  
__________________________________________________________________

## 6. Creating the First Database `university_db`

In [ ]:
!timeout 10s sudo -u postgres psql -tAc "SELECT 1 FROM pg_database WHERE datname='university_db'" | grep -q 1 || sudo -u postgres createdb university_db
!timeout 10s sudo -u postgres psql -c "\l"

## 7. Checking the Connection

In [ ]:
!timeout 10s sudo -u postgres psql -d university_db -c "SELECT current_database(), current_user, inet_server_port();"
!timeout 10s sudo -u postgres psql -d university_db -c "\conninfo"

### Question 3:

Record the parameters:
- **Database:** ______________________
- **User:** __________________________
- **Port:** __________________________
- **Client:** ________________________

## 8. Creating the `students` Table

In [ ]:
!sudo -u postgres psql -d university_db -c " CREATE TABLE IF NOT EXISTS students (student_id SERIAL PRIMARY KEY, full_name VARCHAR(100) NOT NULL, email VARCHAR(120) UNIQUE NOT NULL, program VARCHAR(100), year_of_study INTEGER CHECK (year_of_study BETWEEN 1 AND 6), created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP );"

In [ ]:
!timeout 10s sudo -u postgres psql -d university_db -c "\d students"

### Question 4:

Explain the purpose of the constraints:

- `PRIMARY KEY` — ________________________________________________
- `NOT NULL` — __________________________________________________
- `UNIQUE` — ____________________________________________________
- `CHECK` — _____________________________________________________
- `DEFAULT` — ___________________________________________________

## 9. Adding Test Data

In [ ]:
!sudo -u postgres psql -d university_db -c "INSERT INTO students (full_name, email, program, year_of_study) VALUES ('Aruzhan Sadykova', 'aruzhan@university.kz', 'Cybersecurity', 2), ('Timur Kim', 'timur@university.kz', 'Computer Science', 3), ('Dana Aliyeva', 'dana@university.kz', 'Information Systems', 1) ON CONFLICT (email) DO NOTHING;"

In [ ]:
!timeout 10s sudo -u postgres psql -d university_db -c "SELECT * FROM students ORDER BY student_id;"

### Question 5:

Run the `INSERT` command from section "9. Adding Test Data" again.

What do you see in the response (explain the response)? _____________________

In [ ]:
# QUESTION 6.1:
# Add one more student to the database using the "INSERT ..." command from section 9.
# Write and execute the command in this cell



In [ ]:
# QUESTION 6.2:
# Check what the database looks like using the "SELECT ..." command.
# Write and execute the command in this cell



## 10. Creating the `courses` Table

In [ ]:
!sudo -u postgres psql -d university_db -c " CREATE TABLE IF NOT EXISTS courses (course_id SERIAL PRIMARY KEY, course_name VARCHAR(150) NOT NULL UNIQUE, credits INTEGER NOT NULL CHECK (credits > 0), capacity INTEGER NOT NULL DEFAULT 30 CHECK (capacity > 0));"

In [ ]:
!sudo -u postgres psql -d university_db -c " INSERT INTO courses (course_name, credits, capacity) VALUES ('Server Database Design and Security', 5, 30), ('Database Systems', 5, 35), ('Cybersecurity Fundamentals', 5, 25) ON CONFLICT (course_name) DO NOTHING; "

In [ ]:
!timeout 10s sudo -u postgres psql -d university_db -c "SELECT * FROM courses ORDER BY course_id;"

## 11. Creating the `enrollment` Table

In [ ]:
!sudo -u postgres psql -d university_db -c " CREATE TABLE IF NOT EXISTS enrollment ( enrollment_id SERIAL PRIMARY KEY, student_id INTEGER NOT NULL, course_id INTEGER NOT NULL, enrollment_date DATE DEFAULT CURRENT_DATE, CONSTRAINT fk_enrollmens_student FOREIGN KEY (student_id) REFERENCES students(student_id), CONSTRAINT fk_enrollment_course FOREIGN KEY (course_id) REFERENCES courses(course_id), CONSTRAINT uq_student_course UNIQUE (student_id, course_id));"

In [ ]:
!sudo -u postgres psql -d university_db -c " INSERT INTO enrollment (student_id, course_id) VALUES (1,1), (2,1), (3,2) ON CONFLICT (student_id, course_id) DO NOTHING;"

## 12. First `JOIN`

In [ ]:
!timeout 10s sudo -u postgres psql -d university_db -c " SELECT s.student_id, s.full_name, c.course_name, e.enrollment_date FROM enrollment e JOIN students s ON e.student_id = s.student_id JOIN courses c ON e.course_id = c.course_id ORDER BY s.student_id;"
###
#SELECT
#    s.student_id,
#    s.full_name,
#    c.course_name,
#    e.enrollment_date
#FROM enrollment e
#JOIN students s ON e.student_id = s.student_id
#JOIN courses c ON e.course_id = c.course_id
#ORDER BY s.student_id;
###

### Question 7:
Why are student and course data stored in separate tables rather than in one large table?

**Answer:**  
__________________________________________________________________  
__________________________________________________________________

## 13. Creating a Separate Application User

In [ ]:
!sudo -u postgres psql -c " DO \$\$ BEGIN IF NOT EXISTS (SELECT 1 FROM pg_roles WHERE rolname = 'university_user') THEN CREATE ROLE university_user LOGIN PASSWORD 'StudentLab_2026!'; ELSE ALTER ROLE university_user WITH LOGIN PASSWORD 'StudentLab_2026!'; END IF; END \$\$;"

# DO \$\$
# BEGIN
#    IF NOT EXISTS (SELECT 1 FROM pg_roles WHERE rolname = 'university_user') THEN
#        CREATE ROLE university_user LOGIN PASSWORD 'StudentLab_2026!';
#    ELSE
#        ALTER ROLE university_user WITH LOGIN PASSWORD 'StudentLab_2026!';
#    END IF;
# END
# \$\$;

!sudo -u postgres psql -c "GRANT CONNECT ON DATABASE university_db TO university_user;"

!sudo -u postgres psql -d university_db -c " GRANT USAGE ON SCHEMA public TO university_user; GRANT SELECT, INSERT, UPDATE ON ALL TABLES IN SCHEMA public TO university_user; GRANT USAGE, SELECT ON ALL SEQUENCES IN SCHEMA public TO university_user;"

# GRANT USAGE ON SCHEMA public TO university_user;
# GRANT SELECT, INSERT, UPDATE ON ALL TABLES IN SCHEMA public TO university_user;
# GRANT USAGE, SELECT ON ALL SEQUENCES IN SCHEMA public TO university_user;

In [ ]:
!timeout 10s sudo -u postgres psql -c "\du"

### Question 8: (Security)

Why should a web application not connect to the database using the `postgres` role?

**Answer:**  
__________________________________________________________________  
__________________________________________________________________

## 14. Connecting to PostgreSQL from Python

In [ ]:
!pip -q install psycopg2-binary

In [ ]:
import psycopg2

conn = psycopg2.connect(
    host="localhost",
    port=5432,
    database="university_db",
    user="university_user",
    password="StudentLab_2026!"
)

print("Connection to PostgreSQL was successful.")

## 15. Executing an SQL Query from Python

In [ ]:
cur = conn.cursor()

cur.execute('''
    SELECT student_id, full_name, program, year_of_study
    FROM students
    ORDER BY student_id;
''')

rows = cur.fetchall()
for row in rows:
    print(row)

## 16. Parameterized `INSERT` from Python

In [ ]:
cur.execute('''
    INSERT INTO students (full_name, email, program, year_of_study)
    VALUES (%s, %s, %s, %s)
    ON CONFLICT (email) DO NOTHING;
''', (
    "Ayan Nurgali",
    "ayan@university.kz",
    "Cybersecurity",
    2
))

conn.commit()
print("Data added / existing record retained without changes.")

In [ ]:
cur.execute('''
    SELECT student_id, full_name, email
    FROM students
    ORDER BY student_id;
''')
cur.fetchall()

### Question 9:

Why is parameterization used? (SQL code is separated from user-supplied values)

**Answer:** ___________________________________

## 17. Properly Closing the Connection

In [ ]:
cur.close()
conn.close()
print("Connection closed.")

### Independent Task 1:

Create a `teachers` table with the following fields:
- `teacher_id`
- `full_name`
- `email`
- `department`
- `position`

Complete the following:
1. create the table;
2. add at least **5 teachers**;
3. display all teachers;
4. find teachers from one department;
5. change the position of one teacher;
6. delete one test record;
7. show the table structure.

In [ ]:
# Independent Task 1:
# YOUR CODE FOR THE INDEPENDENT TASK MUST BE PLACED IN THIS CELL
# Example:
# !sudo -u postgres psql -d university_db -c "REQUIRED SQL COMMAND"

### Independent Task 2:

1. Create a `departments` table (`department_id`, `department_name`).
2. Link `teachers` to `departments` through `department_id`.
3. Add a foreign key.
4. Obtain the result `Teacher | Department` using a `JOIN`.

In [ ]:
# Independent Task 2:
# YOUR CODE FOR THE INDEPENDENT TASK MUST BE PLACED IN THIS CELL

# Question 10:

10.1 How does a database differ from a DBMS?  
**Answer:** _______________________________________________

10.2 What is a Database Server?  
**Answer:** _______________________________________________

10.3 What role does `psql` perform?  
**Answer:** _______________________________________________

10.4 Which port does PostgreSQL use by default?  
**Answer:** _______________________________________________

10.5 What does `CREATE DATABASE` do?  
**Answer:** _______________________________________________

10.6 What is a `PRIMARY KEY` used for?  
**Answer:** _______________________________________________

10.7 What is a `FOREIGN KEY` used for?  
**Answer:** _______________________________________________

10.8 How does `postgres` differ from `university_user`?  
**Answer:** _______________________________________________

10.9 Why should an application operate with the minimum necessary privileges?  
**Answer:** _______________________________________________

10.10 How is client-server architecture demonstrated in this laboratory work?  
**Answer:**  
__________________________________________________________________

# Student's Final Conclusion

Write 5–7 sentences about what was completed and what you learned.

**Conclusion:**  
__________________________________________________________________  
__________________________________________________________________  
__________________________________________________________________  
__________________________________________________________________

### What Must Be Submitted to the Instructor:

Save the completed notebook and provide:
- your full name and group;
- answers to all questions in this laboratory work, including the independent tasks and the final conclusion (everything must be completed within this notebook);
- all executed cells must be saved (with their outputs displayed) when submitting the laboratory work (the dates of creation and execution of activities are taken into account);  
- the notebook must be saved under an individual file name.  
**Recommended file name:**  
`Lab01_DSSBD_PostgreSQL_GColab_Surname_Group.ipynb`